In [69]:
import pandas as pd
import numpy as np
import pyshark
import asyncio
import ssl

DATA_FOLDER = "data"
PCAP_FILE = f"{DATA_FOLDER}/trace.pcap"
TOP_URLS_FILE = f"{DATA_FOLDER}/top_urls.csv"

OUTPUT_FOLDER = f"output"
OUTPUT_FILE = f"{OUTPUT_FOLDER}/trace.csv"

In [2]:
top_urls = pd.read_csv(TOP_URLS_FILE)
https_urls = top_urls[top_urls["url"].str.startswith("https://")]
https_urls

,url
2,https://18comic.vip
3,https://7games.bet.br
4,https://8moviesda.net
5,https://9animetv.to
6,https://9moviesda.com
...,...
995,https://zbporn.tv
996,https://zeenews.india.com
997,https://zh.m.wikipedia.org
998,https://zonatmo.com


In [30]:
context = ssl.create_default_context()
i = 1


async def good_url(url):
    domain = url.removeprefix("https://")
    try:
        _, writer = await asyncio.wait_for(asyncio.open_connection(domain, 443), timeout=25)
        await writer.start_tls(
            context, server_hostname=domain, ssl_handshake_timeout=5, ssl_shutdown_timeout=5
        )

        global i
        if i % 50 == 0:
            print(f"{i} good URLs so far...")
        i += 1

        return True
    except Exception as e:
        if str(e):
            print(f"Error connecting {domain}: {e}")
        else:
            print(f"Error connecting {domain}: {repr(e)}")
        return False


good_urls_indices = await asyncio.gather(*[good_url(url) for url in https_urls["url"].tolist()])
good_urls = https_urls[good_urls_indices]

Error connecting bollyflix.tw: [Errno -2] Name or service not known
Error connecting cricbet99.club: [Errno -5] No address associated with hostname
50 good URLs so far...
100 good URLs so far...
150 good URLs so far...
200 good URLs so far...
250 good URLs so far...
Error connecting moviesda.it.com: [Errno -5] No address associated with hostname
300 good URLs so far...
Error connecting new2.filesdl.site: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)
350 good URLs so far...
400 good URLs so far...
Error connecting streamable.cloud: [Errno -2] Name or service not known
Error connecting service.smt.docomo.ne.jp: [SSL: UNSAFE_LEGACY_RENEGOTIATION_DISABLED] unsafe legacy renegotiation disabled (_ssl.c:1028)
450 good URLs so far...
Error connecting smt.docomo.ne.jp: [SSL: UNSAFE_LEGACY_RENEGOTIATION_DISABLED] unsafe legacy renegotiation disabled (_ssl.c:1028)
500 good URLs so far...
Error connecting videq.stream: [Errno -3] T

In [55]:
data = []
good_streams = set()
skipped_packets = 0


def process_server_hello(packet):
    good_streams.add(packet.tcp.stream)


def process_client_hello(packet):
    if packet.tcp.stream not in good_streams:
        global skipped_packets
        skipped_packets += 1
        return
    if not hasattr(packet.tls, "handshake_extensions_server_name"):
        print(f"Skipping packet without SNI")
        return
    data.append(
        {
            "timestamp": packet.sniff_time,
            "domain": packet.tls.handshake_extensions_server_name,
        }
    )


# We only want TLS Client Hello packets, but we want to skip streams for which the TLS
# handshake never concluded
cap_server_hello = pyshark.FileCapture(PCAP_FILE, display_filter="tls.handshake.type == 2")
await cap_server_hello.packets_from_tshark(process_server_hello)

print(f"Found {len(good_streams)} streams with Server Hello packets.")

cap_client_hello = pyshark.FileCapture(PCAP_FILE, display_filter="tls.handshake.type == 1")
await cap_client_hello.packets_from_tshark(process_client_hello)

print(f"{len(data)} TLS streams found. Skipped {skipped_packets} streams that never completed the handshake.")

Found 2518 streams with Server Hello packets.
2828 TLS streams found. Skipped 63 streams that never completed the handshake.


In [ ]:
data = pd.DataFrame(data)
# Express timestamps as second offset since the first packet
initial_time = data["timestamp"].min()
data["timestamp"] = (data["timestamp"] - initial_time).dt.total_seconds()
data

In [60]:
mapping = {}

good_urls = good_urls.sample(frac=1).reset_index(drop=True)
index = 0
for domain in data["domain"].unique():
    mapping[domain] = good_urls["url"].iloc[index]
    index += 1
    if index >= len(good_urls):
        index = 0

data_mapped = data.copy()
data_mapped["url"] = data_mapped["domain"].map(mapping)
data_mapped = data_mapped.drop(columns=["domain"])
data_mapped

,timestamp,url
0,0.000000,https://www.turkiye.gov.tr
1,18.522828,https://tw.stock.yahoo.com
2,24.373063,https://tw.stock.yahoo.com
3,31.596398,https://tw.stock.yahoo.com
4,45.501952,https://tw.stock.yahoo.com
...,...,...
2823,59834.086450,https://www.xvideos.es
2824,59834.161956,https://tw.stock.yahoo.com
2825,59834.454421,https://tw.stock.yahoo.com
2826,59834.611928,https://poki.com


In [61]:
BUCKET_SIZE = 30 * 60  # 30 mins

data_grouped = data_mapped
data_grouped["group"] = (data_grouped["timestamp"] // BUCKET_SIZE).astype(int)
data_grouped.iloc[np.r_[0:5, -5:0]]

,timestamp,url,group
0,0.000000,https://www.turkiye.gov.tr,0
1,18.522828,https://tw.stock.yahoo.com,0
2,24.373063,https://tw.stock.yahoo.com,0
3,31.596398,https://tw.stock.yahoo.com,0
4,45.501952,https://tw.stock.yahoo.com,0
2823,59834.086450,https://www.xvideos.es,33
2824,59834.161956,https://tw.stock.yahoo.com,33
2825,59834.454421,https://tw.stock.yahoo.com,33
2826,59834.611928,https://poki.com,33
2827,59834.785061,https://tw.stock.yahoo.com,33


In [66]:
BUCKETS_TO_KEEP = 8

# we will keep the most populous buckets
group_sizes = data_grouped.groupby("group").size()
data_pruned = data_grouped[
    data_grouped["group"].isin(group_sizes.nlargest(BUCKETS_TO_KEEP).index.tolist())
]
print(f"Largest bucket has {data_pruned.groupby('group').size().max()} packets")
print(f"Smallest bucket has {data_pruned.groupby('group').size().min()} packets")
data_pruned

Largest bucket has 404 packets
Smallest bucket has 94 packets


,timestamp,url,group
217,7203.774100,https://tw.stock.yahoo.com,4
218,7219.929907,https://m.daum.net,4
219,7243.855177,https://kpqz.brdbuxte.com,4
220,7244.002885,https://news.google.com,4
221,7244.050038,https://kpqz.brdbuxte.com,4
...,...,...,...
1943,26652.982861,https://finance.yahoo.com,14
1944,26666.797080,https://kingbokep.pro,14
1945,26667.092544,https://litnet.com,14
1946,26667.109322,https://fastdl.app,14


In [67]:
data_pruned.rename(columns={"group": "trace"}).to_csv(OUTPUT_FILE, index=False)

In [48]:
def remap_trace_index(traces):
    mapping = {}
    for i, trace_id in enumerate(traces["trace"].unique()):
        mapping[trace_id] = i
    traces["trace"] = traces["trace"].map(mapping)

def combine_traces(trace_file_1, trace_file_2):
    trace_1 = pd.read_csv(trace_file_1)
    remap_trace_index(trace_1)
    trace_2 = pd.read_csv(trace_file_2)
    remap_trace_index(trace_2)
    offset = trace_1["trace"].max() + 1
    trace_2["trace"] += offset
    return pd.concat([trace_1, trace_2], ignore_index=True)